# Functional Pipeline Walkthrough

This notebook walks you through the functional preprocessing pipeline step by step. Each section explains **what** is happening and **why**, followed by a bash cell you can run directly.

### What this pipeline produces
Starting from raw BOLD runs (in BIDS format), we end up with:
- Susceptibility-distortion-corrected BOLD timeseries
- Motion-corrected BOLD data coregistered to the FreeSurfer T1
- BOLD timeseries projected onto the cortical surface (GIFTI format)
- A merged confounds table (motion + fMRIPrep noise regressors)
- Denoised BOLD (volumetric + surface) ready for pRF/CF fitting

### Pipeline stages at a glance
| Stage | Script | Tool | What it does |
|-------|--------|------|--------------|
| 1 | `s01_sdc_AFNI.py` | AFNI | Susceptibility distortion correction |
| 2 | `s02_coreg.py` | FSL + FreeSurfer | Motion correction, coregistration, surface projection |
| 3 | `s03_fmriprep_func.py` | fMRIPrep | Extract noise confounds |
| 4 | `s04_confounds.py` | Python | Merge confounds, PCA, SG filter, denoise |

### Prerequisites
Before starting:
- The **anatomical pipeline** must have been completed (FreeSurfer reconstruction done)
- `PIPELINE_DIR` is set in your `~/.bash_profile`
- Docker or Apptainer/Singularity is available (for AFNI, FSL, FreeSurfer, fMRIPrep)
- Your `config/project_<name>.sh` file is set up
- BOLD data and fieldmaps are in BIDS format under `$BIDS_DIR`

---

## 0. Set your variables

Fill in the cell below then **run it once**. It sets `PROJECT`, `SUB`, `SES`, and `TASK` as environment variables that every subsequent bash cell will inherit automatically.

In [3]:
import subprocess, os
from cvl_utils.preproc_func import set_project
# ── Fill these in ────────────────────────────────────────────────────────────
PROJECT = 'lhon'    # matches your config/project_<name>.sh file
SUB     = 'sub-C001'  # subject label, e.g. sub-01
SES     = 'ses-prf'  # session containing the T1w scan, e.g. ses-01
TASK    = 'prfNonEq'
# ─────────────────────────────────────────────────────────────────────────────
# Source the project config once and capture every exported variable into
# os.environ so all %%bash cells below pick them up without re-sourcing.
set_project(PROJECT)
os.environ['SUB']  = SUB
os.environ['SES']  = SES
os.environ['TASK'] = TASK

print(f"Project:     {os.environ.get('PROJ_NAME')}")
print(f"BIDS dir:    {os.environ.get('BIDS_DIR')}")
print(f"FreeSurfer:  {os.environ.get('SUBJECTS_DIR')}")
print(f"AFNI image:  {os.environ.get('AFNI_IMAGE')}")
print(f"FSL image:   {os.environ.get('FSL_FREESURFER_IMAGE')}")
print(f"Subject:     {os.environ.get('SUB')}")
print(f"Session:     {os.environ.get('SES')}")
print(f"Task:        {os.environ.get('TASK') or '<all tasks>'}")

Project:     lhon
BIDS dir:    /Users/marcusdaghlian/projects/lhon_sl
FreeSurfer:  /Users/marcusdaghlian/projects/lhon_sl/derivatives/freesurfer
CTX dir:     (not set)
Project:     lhon
BIDS dir:    /Users/marcusdaghlian/projects/lhon_sl
FreeSurfer:  /Users/marcusdaghlian/projects/lhon_sl/derivatives/freesurfer
AFNI image:  vnmd/afni_26.0.07
FSL image:   ndock_fsl_freesurfer:latest
Subject:     sub-C001
Session:     ses-prf
Task:        prfNonEq


---
## Step 1 — Susceptibility Distortion Correction (AFNI)

### What is susceptibility distortion?
EPI (echo-planar imaging) BOLD sequences are fast, but they accumulate phase errors wherever there are local magnetic field inhomogeneities — typically at air-tissue interfaces (sinuses, ear canals, skull base). These field inhomogeneities cause the image to be stretched or compressed along the **phase-encoding (PE) axis**, making it look like the brain has been squeezed or pulled in one direction. Regions like inferior prefrontal cortex and orbitofrontal cortex are particularly affected.

### How does this correction work?
The simplest and most robust correction strategy is to acquire a second short EPI scan with the **opposite phase-encoding direction** (often called a "topup" or "reverse-PE" scan). Because the distortion flips sign when you reverse the PE direction, comparing the two images lets you estimate the underlying field map. AFNI's `unWarpEPIfloat.py` does this estimation and then applies the resulting warp to undistort the full BOLD timeseries.

For each run the pipeline does:
1. Convert BOLD and reverse-PE images to AFNI format (`3dcopy`)
2. Run `unWarpEPIfloat.py` — this estimates the field map from the forward/reverse pair and applies the warp to the BOLD timeseries
3. Apply the same warp to the single-band reference (`sbref`) — a higher-SNR single volume acquired before each run that we will later use as a motion-correction target

### What do I need in my BIDS directory?
For each BOLD run (`func/sub-XX_ses-XX_task-XX_run-XX_bold.nii.gz`) you need:
- A matching `sbref` in `func/` (`*sbref.nii.gz`)
- A matching reverse-PE EPI in `fmap/` (`*_epi.nii.gz` with `IntendedFor` pointing to the BOLD run)

### Outputs (saved to `derivatives/s1_AFNI_sdc/`)
- `*_sdc_bold.nii.gz` — distortion-corrected BOLD timeseries
- `*_sdc_sbref.nii.gz` — distortion-corrected sbref image

In [4]:
%%bash
# ── Option A: Run locally ─────────────────────────────────────────────────────
# Runs SDC on your local machine using the AFNI Docker container.
# Remove --task and the value to process all tasks in the session.

$PYPACKAGE_MANAGER run -n preproc \
    python "${PIPELINE_DIR}/functional/s01_sdc_AFNI.py" \
        --bids-dir    "${BIDS_DIR}" \
        --sub         "${SUB}" \
        --ses         "${SES}" \
        --task        "${TASK}" \
        --output-file s1_AFNI_sdc_TEST \
        --afni-docker "${AFNI_IMAGE}"

TypeError: %d format: a real number is required, not NoneType

TypeError: %d format: a real number is required, not NoneType


CondaError: KeyboardInterrupt



In [5]:
%%bash
# ── Check SDC outputs ─────────────────────────────────────────────────────────
echo "=== SDC outputs for ${SUB} / ${SES} ==="
ls "${BIDS_DIR}/derivatives/s1_AFNI_sdc/${SUB}/${SES}/" 2>/dev/null \
    | grep -E '(bold|sbref)\.nii' \
    || echo "  No outputs found yet."

=== SDC outputs for sub-C001 / ses-prf ===
sub-C001_ses-prf_task-prfNonEq_run-1_sdc_bold.nii.gz
sub-C001_ses-prf_task-prfNonEq_run-1_sdc_sbref.nii.gz
sub-C001_ses-prf_task-prfNonEq_run-2_sdc_bold.nii.gz
sub-C001_ses-prf_task-prfNonEq_run-2_sdc_sbref.nii.gz
sub-C001_ses-prf_task-prfNonEq_run-3_sdc_bold.nii.gz
sub-C001_ses-prf_task-prfNonEq_run-3_sdc_sbref.nii.gz


---
## Step 2 — Motion Correction, Coregistration & Surface Projection (FSL + FreeSurfer)

### Why do we need this step?
Even small head movements during scanning (< 1 mm) cause signal fluctuations that are correlated across voxels, mimicking real neural signals. We need to:
1. **Motion correct** — realign every BOLD volume back to a common reference so that a given voxel always corresponds to the same piece of brain
2. **Coregister to anatomy** — map the BOLD data from the native functional space into the FreeSurfer T1 space so we know which cortical surface vertex each voxel corresponds to
3. **Project to the cortical surface** — sample the volumetric BOLD data onto the FreeSurfer surface mesh, averaging across cortical depth; this is required for pRF/CF fitting and pycortex visualisation

### Coregistration strategy
A key challenge is that we have multiple runs, potentially across multiple sessions, and we want them all in a *consistent* space. The strategy is:

1. **BREF_MAIN** — one high-quality reference image per subject (the first sbref, or vol-0 of the first BOLD run). This becomes the single anchor point for the whole subject.
2. **bbregister** — FreeSurfer's boundary-based registration aligns `BREF_MAIN` to the T1 anatomy with sub-voxel accuracy. This expensive step is only done once per subject.
3. **sbref_i → BREF_MAIN** — each run's own sbref is registered to `BREF_MAIN` with a 6-DOF rigid-body FLIRT. This is cheap and handles session-to-session head repositioning.
4. **MCFLIRT per run** — FSL's MCFLIRT realigns every BOLD volume to its run-specific sbref (sbref_i). This gives per-volume motion parameter estimates.
5. **Chain the transforms** — the three transforms (`VOL → sbref_i`, `sbref_i → BREF_MAIN`, `BREF_MAIN → FS_T1`) are concatenated so that the final resampling is done in *one single interpolation step*. One interpolation = less blurring than applying three separate warps.
6. **Surface projection** — `mri_vol2surf` samples the coregistered BOLD volume onto the lh/rh cortical surfaces, averaging across 20–80% cortical depth in steps of 10%.

### What do I need?
- Completed anatomical pipeline (FreeSurfer reconstruction in `$SUBJECTS_DIR`)
- SDC outputs from Step 1 in `derivatives/s1_AFNI_sdc/`

### Outputs (saved to `derivatives/s2_coreg/`)
- `*_space-fsT1_desc-moco_bbreg_bold.nii.gz` — BOLD in FreeSurfer T1 space, motion corrected
- `*_space-fsnative_hemi-L_bold.func.gii` / `*_hemi-R_bold.func.gii` — surface BOLD (left/right hemisphere)
- `*_desc-mcflirt_motion.par` — 6-DOF motion parameters per volume (for confound regression)
- `*_desc-mcflirt+bbreg_transforms/` — per-volume combined transform matrices

> **Environment note:** FSL and FreeSurfer are run inside a container. The image is set by `$FSL_FREESURFER_IMAGE` in your project config.

In [6]:
%%bash
# ── Option A: Run locally ─────────────────────────────────────────────────────
# Processes all runs for the given subject + session.
# This can take 20-60 min per run depending on run length and hardware.

$PYPACKAGE_MANAGER run -n preproc \
    python "${PIPELINE_DIR}/functional/s02_coreg.py" \
        --bids-dir    "${BIDS_DIR}" \
        --sub         "${SUB}" \
        --ses         "${SES}" \
        --input-file  s1_AFNI_sdc \
        --output-file s2_coreg_TEST \
        --subjects-dir "${SUBJECTS_DIR}" \
        --docker      "${FSL_FREESURFER_IMAGE}"

TypeError: %d format: a real number is required, not NoneType

TypeError: %d format: a real number is required, not NoneType


CondaError: KeyboardInterrupt



In [23]:
%%bash
# ── Check coreg outputs ───────────────────────────────────────────────────────
echo "=== Volumetric BOLD (fsT1 space) ==="
ls "${BIDS_DIR}/derivatives/s2_coreg/${SUB}/${SES}/"*space-fsT1*.nii* 2>/dev/null \
    || echo "  none found"

echo ""
echo "=== Surface BOLD (GIFTI) ==="
ls "${BIDS_DIR}/derivatives/s2_coreg/${SUB}/${SES}/"*.func.gii 2>/dev/null \
    || echo "  none found"

echo ""
echo "=== Motion parameters (.par) ==="
ls "${BIDS_DIR}/derivatives/s2_coreg/${SUB}/${SES}/"*.par 2>/dev/null \
    || echo "  none found"

=== Volumetric BOLD (fsT1 space) ===


/Users/marcusdaghlian/projects/lhon_sl/derivatives/s2_coreg/sub-C001/ses-prf/sub-C001_ses-prf_task-prfNonEq_run-1_space-fsT1_desc-moco_bbreg_bold.nii.gz
/Users/marcusdaghlian/projects/lhon_sl/derivatives/s2_coreg/sub-C001/ses-prf/sub-C001_ses-prf_task-prfNonEq_run-2_space-fsT1_desc-moco_bbreg_bold.nii.gz
/Users/marcusdaghlian/projects/lhon_sl/derivatives/s2_coreg/sub-C001/ses-prf/sub-C001_ses-prf_task-prfNonEq_run-3_space-fsT1_desc-moco_bbreg_bold.nii.gz

=== Surface BOLD (GIFTI) ===
/Users/marcusdaghlian/projects/lhon_sl/derivatives/s2_coreg/sub-C001/ses-prf/sub-C001_ses-prf_task-prfNonEq_run-1_space-fsnative_hemi-L_bold.func.gii
/Users/marcusdaghlian/projects/lhon_sl/derivatives/s2_coreg/sub-C001/ses-prf/sub-C001_ses-prf_task-prfNonEq_run-1_space-fsnative_hemi-R_bold.func.gii
/Users/marcusdaghlian/projects/lhon_sl/derivatives/s2_coreg/sub-C001/ses-prf/sub-C001_ses-prf_task-prfNonEq_run-2_space-fsnative_hemi-L_bold.func.gii
/Users/marcusdaghlian/projects/lhon_sl/derivatives/s2_coreg/s

### QC — Check coregistration

Before going further, it is worth checking that the coregistration looks correct. Open `BREF_MAIN_aligned.nii.gz` (found in `derivatives/s2_coreg/sub-XX/`) alongside the FreeSurfer T1 (`desc-fsbrain.nii.gz`) in freeview or FSLeyes and overlay the surfaces. The BOLD coverage region should align well with the T1.

Key things to check:
- The functional image is not shifted relative to the anatomy (no gross misregistration)
- The susceptibility-distorted regions (inferior frontal cortex, temporal poles) look reasonable — some residual distortion is normal, but there should be no large geometric deformation remaining
- The motion parameters (`.par` files) do not show any sudden large jumps > 2 mm / 2° which would indicate a problem with a specific run

---
## Step 3 — fMRIPrep (confounds only)

### Why run fMRIPrep again?
We already did our own susceptibility distortion correction and motion correction in Steps 1 and 2 — we do **not** want fMRIPrep to redo those. However, fMRIPrep is excellent at extracting physiological noise confounds (aCompCor, tCompCor, CSF, white matter signals) that we want to use in Step 4 to denoise the data.

The trick is that we feed fMRIPrep our **already preprocessed** BOLD data (the motion-corrected, coregistered volumes from Step 2) and tell it to:
- `--ignore fieldmaps` — don't try to do SDC (we already did it)
- `--ignore slicetiming` — don't do slice timing correction
- `--output-spaces func` — keep outputs in native (functional) space
- `--fs-subjects-dir` — point to our existing FreeSurfer reconstruction so it doesn't rerun surface reconstruction

fMRIPrep then runs its noise estimation pipeline (aCompCor on white matter and CSF ROIs, tCompCor, global signals, etc.) and produces a `*_desc-confounds_timeseries.tsv` per run.

### What is aCompCor?
aCompCor (anatomical component correction) extracts the principal components of signal variation within white matter and CSF masks. These regions should not have BOLD signal from neural activity, so any fluctuations there are noise — scanner thermal noise, cardiac pulsatility, respiratory effects. Regressing out the top PCA components from these masks removes a large portion of structured noise.

### What do I need?
- Step 2 outputs in `derivatives/s2_coreg/`
- `CONTAINER_TYPE` set in your project config (`docker` or `apptainer`)
- `FPREP_IMAGE` (Docker) or `FPREP_SIF` + `SIF_DIR` (Apptainer) set in your project config

### Outputs (saved to `derivatives/fmriprep/`)
- `*_desc-confounds_timeseries.tsv` — noise regressors per run (many columns)
- `*_desc-brain_mask.nii.gz` — brain mask (used for DVARS in Step 4)

> **Note:** fMRIPrep is staged through a temporary `derivatives/FPREP_BIDS/` folder. The script copies the Step 2 BOLD files in there with clean BIDS names and writes minimal JSON sidecars, then runs fMRIPrep on that folder.

In [16]:
%%bash
# Quickly backup the fmriprep folder so as not to overwrite it
cp -r $BIDS_DIR/derivatives/fmriprep/${SUB} $BIDS_DIR/derivatives/fmriprep/${SUB}_BU
ls $BIDS_DIR/derivatives/fmriprep/

sub-C001
sub-C001_BU


In [17]:
%%bash
# ── Option A: Run locally ─────────────────────────────────────────────────────
# fMRIPrep will run in a container. It needs a FreeSurfer license at
# $PIPELINE_DIR/config/license.txt



$PYPACKAGE_MANAGER run -n preproc \
    python "${PIPELINE_DIR}/functional/s03_fmriprep_func.py" \
        --bids-dir   "${BIDS_DIR}" \
        --sub        "${SUB}" \
        --ses        "${SES}" \
        --input-file s2_coreg

TypeError: %d format: a real number is required, not NoneType

TypeError: %d format: a real number is required, not NoneType


CondaError: KeyboardInterrupt



In [24]:
%%bash
# ── Check fMRIPrep confound files ─────────────────────────────────────────────
echo "=== Confounds timeseries TSV files ==="
ls "${BIDS_DIR}/derivatives/fmriprep/${SUB}/${SES}/func/"*confounds_timeseries.tsv 2>/dev/null \
    || echo "  none found"

echo ""
echo "=== Brain masks ==="
ls "${BIDS_DIR}/derivatives/fmriprep/${SUB}/${SES}/func/"*brain_mask.nii* 2>/dev/null \
    || echo "  none found"

=== Confounds timeseries TSV files ===


/Users/marcusdaghlian/projects/lhon_sl/derivatives/fmriprep/sub-C001/ses-prf/func/sub-C001_ses-prf_task-prfNonEq_run-1_desc-confounds_timeseries.tsv
/Users/marcusdaghlian/projects/lhon_sl/derivatives/fmriprep/sub-C001/ses-prf/func/sub-C001_ses-prf_task-prfNonEq_run-2_desc-confounds_timeseries.tsv
/Users/marcusdaghlian/projects/lhon_sl/derivatives/fmriprep/sub-C001/ses-prf/func/sub-C001_ses-prf_task-prfNonEq_run-3_desc-confounds_timeseries.tsv

=== Brain masks ===
/Users/marcusdaghlian/projects/lhon_sl/derivatives/fmriprep/sub-C001/ses-prf/func/sub-C001_ses-prf_task-prfNonEq_run-1_desc-brain_mask.nii.gz
/Users/marcusdaghlian/projects/lhon_sl/derivatives/fmriprep/sub-C001/ses-prf/func/sub-C001_ses-prf_task-prfNonEq_run-2_desc-brain_mask.nii.gz
/Users/marcusdaghlian/projects/lhon_sl/derivatives/fmriprep/sub-C001/ses-prf/func/sub-C001_ses-prf_task-prfNonEq_run-3_desc-brain_mask.nii.gz


---
## Step 4 — Confound Extraction & Denoising

### The problem: structured noise in BOLD data
Even after motion correction, BOLD timeseries contain substantial non-neural fluctuations:
- **Cardiac pulsatility** — the heart beat introduces ~1 Hz oscillations (aliased to slow frequencies at typical TRs)
- **Respiratory noise** — breathing causes global signal changes and chest-wall motion
- **Scanner drift** — slow drifts in the baseline signal over the course of a run (minutes-scale)
- **Residual motion artefacts** — sudden displacement spikes and the signal changes that persist after them

The goal of this step is to remove as much of this structured noise as possible while preserving the task-driven BOLD signal.

### What the pipeline does

**Steps 1–3 (confound assembly):**
- Take the fMRIPrep `*_desc-confounds_timeseries.tsv` and **drop all motion columns** from it (we'll replace those with our own, FSL-derived versions). What we keep from fMRIPrep: aCompCor, tCompCor, CSF and white matter mean signals.
- Load the MCFLIRT `.par` file and compute:
  - 6 rigid-body motion parameters (3 rotations, 3 translations)
  - Framewise Displacement (FD) — the summed absolute movement between consecutive volumes
  - DVARS — the RMS change in BOLD signal across the brain between consecutive volumes
- Merge everything into a single `*_desc-confounds.tsv` per run

**Step 4 (PCA):**
- Take a subset of the confound columns (motion params, FD, DVARS, aCompCor, tCompCor, CSF, WM)
- Z-score them, run PCA, retain the top N components (default: 6)
- Apply a **Savitzky-Golay high-pass filter** to each PCA component — this removes slow scanner drift from the regressors themselves, so we don't inadvertently remove low-frequency neural signal when we regress them out

**Step 5 (denoising):**
- Apply the same SG high-pass filter to the BOLD data (removes scanner drift)
- Regress the PCA confound components out of every voxel/vertex timeseries using OLS
- Save denoised volumetric BOLD and denoised surface GIFTI files

### Savitzky-Golay high-pass filter
Rather than a hard-cutoff Fourier filter (which can introduce ringing artefacts), we use a Savitzky-Golay filter to estimate and subtract the low-frequency trend in each timeseries. The `--sg-window` parameter (in seconds) controls the cutoff: a 347s window means fluctuations slower than ~347s are treated as drift and removed. The polynomial order (`--sg-order`, default 3) controls how smoothly the trend is estimated.

### Filter-only mode
You can run with `--filter-only` to apply only the SG high-pass filter, skipping the confound regression entirely. This is useful for comparing how much of the denoising benefit comes from filtering vs. from the PCA regression.

### Outputs (saved to `derivatives/s4_denoised/`)
- `*_desc-confounds.tsv` — merged confound table (motion + fMRIPrep noise)
- `*_desc-pca_confounds.tsv` — top PCA components (high-pass filtered)
- `*_desc-denoised_bold.nii.gz` — denoised volumetric BOLD
- `*_space-fsnative_hemi-L_desc-denoised_bold.func.gii` — denoised left hemisphere surface
- `*_space-fsnative_hemi-R_desc-denoised_bold.func.gii` — denoised right hemisphere surface

In [22]:
%%bash
# ── Option A: Run locally (full denoising) ────────────────────────────────────
# Uses 6 PCA components and a 347s Savitzky-Golay high-pass window by default.
# Adjust --n-pca, --sg-window, --sg-order if needed.

$PYPACKAGE_MANAGER run -n preproc \
    python "${PIPELINE_DIR}/functional/s04_confounds.py" \
        --bids-dir    "${BIDS_DIR}" \
        --sub         "${SUB}" \
        --ses         "${SES}" \
        --moco-file   s2_coreg \
        --output-file s4_denoised_test \
        --n-pca       6 \
        --sg-window   347 \
        --sg-polyorder    3 \
        --skip denoise_vol

-------------------------------------------------------
Processing: Confound Extraction + Denoising
-------------------------------------------------------
 Moco input  : /Users/marcusdaghlian/CVL Dropbox/Marcus  Daghlian/CVL/CVL Meetings/TrainingSessions_CVL/150626_CVL_MRI/BIDS/derivatives/s2_coreg
 fMRIprep    : /Users/marcusdaghlian/CVL Dropbox/Marcus  Daghlian/CVL/CVL Meetings/TrainingSessions_CVL/150626_CVL_MRI/BIDS/derivatives/fmriprep
 Output      : /Users/marcusdaghlian/CVL Dropbox/Marcus  Daghlian/CVL/CVL Meetings/TrainingSessions_CVL/150626_CVL_MRI/BIDS/derivatives/s4_denoised_test
 Subject     : sub-C001
 Session     : ses-prf
 PCA comps   : 6
 SG window   : 347s (order 3)
 Force-skip  : denoise_vol
-------------------------------------------------------

Found 3 BOLD run(s).
  - sub-C001_ses-prf_task-prfNonEq_run-1_space-fsT1_desc-moco_bbreg_bold.nii.gz
  - sub-C001_ses-prf_task-prfNonEq_run-2_space-fsT1_desc-moco_bbreg_bold.nii.gz
  - sub-C001_ses-prf_task-prfNonEq_run-3_s

Traceback (most recent call last):
  File "/Users/marcusdaghlian/projects/dp-clean-link/240522NG/hypot/code/hypot_code//functional/s04_confounds.py", line 1161, in <module>
    main()
  File "/Users/marcusdaghlian/projects/dp-clean-link/240522NG/hypot/code/hypot_code//functional/s04_confounds.py", line 1144, in main
    run_pipeline(
  File "/Users/marcusdaghlian/projects/dp-clean-link/240522NG/hypot/code/hypot_code//functional/s04_confounds.py", line 1018, in run_pipeline
    run_results = process_run(
                  ^^^^^^^^^^^^
  File "/Users/marcusdaghlian/projects/dp-clean-link/240522NG/hypot/code/hypot_code//functional/s04_confounds.py", line 670, in process_run
    build_confounds_tsv(
  File "/Users/marcusdaghlian/projects/dp-clean-link/240522NG/hypot/code/hypot_code//functional/s04_confounds.py", line 275, in build_confounds_tsv
    motion_df = extract_mcflirt_confounds(
                ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/marcusdaghlian/projects/dp-clean-link/240522NG

CalledProcessError: Command 'b'# \xe2\x94\x80\xe2\x94\x80 Option A: Run locally (full denoising) \xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\n# Uses 6 PCA components and a 347s Savitzky-Golay high-pass window by default.\n# Adjust --n-pca, --sg-window, --sg-order if needed.\n\n$PYPACKAGE_MANAGER run -n preproc \\\n    python "${PIPELINE_DIR}/functional/s04_confounds.py" \\\n        --bids-dir    "${BIDS_DIR}" \\\n        --sub         "${SUB}" \\\n        --ses         "${SES}" \\\n        --moco-file   s2_coreg \\\n        --output-file s4_denoised_test \\\n        --n-pca       6 \\\n        --sg-windo

In [ ]:
%%bash
# ── Option A (filter-only variant) ───────────────────────────────────────────
# Skip confound regression — apply only the SG high-pass filter.
# Outputs are saved as _desc-filtered_bold so they can be compared against
# _desc-denoised_bold to isolate how much the PCA regression helps.

$PYPACKAGE_MANAGER run -n preproc \
    python "${PIPELINE_DIR}/functional/s04_confounds.py" \
        --bids-dir    "${BIDS_DIR}" \
        --sub         "${SUB}" \
        --ses         "${SES}" \
        --moco-file   s2_coreg \
        --output-file s4_denoised \
        --sg-window   347 \
        --sg-order    3 \
        --filter-only

In [ ]:
%%bash
# ── Option B: Submit to HPC cluster ──────────────────────────────────────────

bash "${PIPELINE_DIR}/functional/s04_confounds_hpc.sh" \
    --bids-dir    "${BIDS_DIR}" \
    --sub         "${SUB}" \
    --ses         "${SES}" \
    --moco-file   s2_coreg \
    --output-file s4_denoised

In [ ]:
%%bash
# ── Check denoising outputs ───────────────────────────────────────────────────
echo "=== Denoised volumetric BOLD ==="
ls "${BIDS_DIR}/derivatives/s4_denoised/${SUB}/${SES}/"*denoised_bold.nii* 2>/dev/null \
    || echo "  none found"

echo ""
echo "=== Denoised surface BOLD ==="
ls "${BIDS_DIR}/derivatives/s4_denoised/${SUB}/${SES}/"*denoised_bold.func.gii 2>/dev/null \
    || echo "  none found"

echo ""
echo "=== Confound tables ==="
ls "${BIDS_DIR}/derivatives/s4_denoised/${SUB}/${SES}/"*.tsv 2>/dev/null \
    || echo "  none found"

### QC — Inspect the confounds

Before proceeding to pRF/CF fitting it is worth visualising the confound timeseries to check:
- Motion parameters don't show sudden large jumps (spikes > 1–2 mm / °)
- FD is generally low (< 0.5 mm) — high FD runs may need to be excluded
- The PCA components look like reasonable noise timeseries (not dominated by the task HRF)

The cell below plots a quick summary of the motion and FD for every run.

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

bids_dir = os.environ.get('BIDS_DIR', '')
sub      = os.environ.get('SUB', '')
ses      = os.environ.get('SES', '')

conf_files = sorted(glob.glob(
    os.path.join(bids_dir, 'derivatives', 's4_denoised', sub, ses,
                 '*_desc-confounds.tsv')))

if not conf_files:
    print('No confound files found — run Step 4 first.')
else:
    fig, axes = plt.subplots(len(conf_files), 2,
                             figsize=(14, 3 * len(conf_files)),
                             squeeze=False)
    for i, cf in enumerate(conf_files):
        run_name = os.path.basename(cf)
        df = pd.read_csv(cf, sep='\t', na_values='n/a')

        # Motion parameters
        ax = axes[i, 0]
        for col in ['trans_x', 'trans_y', 'trans_z']:
            if col in df: ax.plot(df[col], label=col, lw=0.8)
        ax.set_title(run_name, fontsize=8)
        ax.set_ylabel('Translation (mm)')
        ax.legend(fontsize=6, loc='upper right')
        ax.axhline(0, color='k', lw=0.5)

        # FD
        ax2 = axes[i, 1]
        if 'framewise_displacement' in df:
            ax2.plot(df['framewise_displacement'], color='orange', lw=0.8)
            ax2.axhline(0.5, color='red', lw=1, linestyle='--', label='0.5 mm')
            ax2.set_ylabel('FD (mm)')
            ax2.legend(fontsize=6)

    plt.tight_layout()
    plt.show()

---
## Summary: Output Files

After completing all four stages your derivatives directory will look something like this:

```
$BIDS_DIR/derivatives/
├── s1_AFNI_sdc/
│   └── sub-01/ses-01/
│       ├── task-pRF_run-01/             ← working directory (AFNI intermediates)
│       ├── *_task-pRF_run-01_sdc_bold.nii.gz   ← distortion-corrected BOLD
│       └── *_task-pRF_run-01_sdc_sbref.nii.gz  ← distortion-corrected sbref
│
├── s2_coreg/
│   └── sub-01/
│       ├── sub-01_BREF_MAIN.nii.gz       ← subject-level reference image
│       ├── sub-01_desc-fsbrain.nii.gz    ← FreeSurfer T1 (converted to NIfTI)
│       ├── sub-01_desc-sbref2fs_bbr.dat  ← bbregister registration
│       └── ses-01/
│           ├── *_space-fsT1_desc-moco_bbreg_bold.nii.gz  ← BOLD in FS T1 space
│           ├── *_space-fsnative_hemi-L_bold.func.gii     ← LH surface BOLD
│           ├── *_space-fsnative_hemi-R_bold.func.gii     ← RH surface BOLD
│           └── *_desc-mcflirt_motion.par                 ← motion parameters
│
├── fmriprep/
│   └── sub-01/ses-01/func/
│       ├── *_desc-confounds_timeseries.tsv  ← fMRIPrep noise regressors
│       └── *_desc-brain_mask.nii.gz         ← brain mask
│
└── s4_denoised/
    └── sub-01/ses-01/
        ├── *_desc-confounds.tsv          ← merged confound table
        ├── *_desc-pca_confounds.tsv      ← PCA noise components (SG filtered)
        ├── *_desc-denoised_bold.nii.gz   ← denoised volumetric BOLD  ← USE THIS
        ├── *_hemi-L_desc-denoised_bold.func.gii  ← denoised LH surface  ← USE THIS
        └── *_hemi-R_desc-denoised_bold.func.gii  ← denoised RH surface  ← USE THIS
```

### What comes next?

The denoised surface GIFTIs (`*_desc-denoised_bold.func.gii`) are the primary input to the pRF and connective field (CF) fitting notebooks:
- **pRF fitting** — `postproc/s01_gauss_prfpy.py` (or `s01_gauss_prfpy.ipynb` for interactive use)
- **CF fitting** — `postproc/s03_cf_prfpy_hpc.sh`

Refer to the `postproc/` walkthrough notebook for those steps.